# Integrated Gradients cross-check of the OLL screen

Our out-of-lung-localization (OLL) screen uses Grad-CAM. This checks whether the finding is **method-specific** by recomputing OLL with a second attribution method, Integrated Gradients (IG), and comparing per-disease. If the two agree, the localization result isn't a Grad-CAM artifact (cf. Adebayo et al., sanity checks for saliency).

**Both methods run on the SAME GPU here**, so the comparison isolates method, not device (we already know OLL is device-sensitive, so we hold device fixed).

**Before running:** `Runtime → Change runtime type → GPU`.

**You need** `chexpert_data.zip` from Drive (images are gitignored). IG is ~`n_steps`× the cost of Grad-CAM, which is why this runs on GPU.

**Send back:** `oll_crosscheck.zip` (the two OLL CSVs: Grad-CAM and IG).

### 1. Clone + install (captum is the new dep)

In [ ]:
!git clone -b feat/extra-experiments https://github.com/su-andrew/cs229-shortcut-detection.git
%cd cs229-shortcut-detection
!pip install -q torch torchvision torchxrayvision grad-cam scikit-image scikit-learn pandas numpy matplotlib pyyaml captum

### 2. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))

### 3. Mount Drive + unzip data
Edit `DATA_ZIP` to its path in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS to your data zip's Drive path >>>
DATA_ZIP = '/content/drive/MyDrive/chexpert_data.zip'

!unzip -q "$DATA_ZIP" -d data/
import os
need = ['data/chexpert/PNG_valid', 'data/chexpert/metadata.csv',
        'results/predictions_val.csv']
# predictions_val.csv ships in the repo? if not, regenerate the baseline first.
missing = [p for p in need if not os.path.exists(p)]
print('missing:', missing if missing else 'none')
assert os.path.exists('data/chexpert/PNG_valid'), 'data not unzipped — check DATA_ZIP'

### 4. Ensure baseline predictions exist
OLL needs `results/predictions_val.csv`. If it's not in the repo clone, regenerate it (cheap, inference-only).

In [ ]:
import os
if not os.path.exists('results/predictions_val.csv'):
    !python -m src.baseline --device cuda
print('predictions ready:', os.path.exists('results/predictions_val.csv'))

### 5. OLL with Grad-CAM and with IG, all 4 labels, same GPU

In [ ]:
!python -m src.ola --num-labels 4 --device cuda --method gradcam
!python -m src.ola --num-labels 4 --device cuda --method ig

### 6. Compare — does IG agree with Grad-CAM?

In [ ]:
import pandas as pd
g = pd.read_csv('results/oll_by_disease.csv').set_index('label')['oll_pos_mean']
i = pd.read_csv('results/oll_by_disease_ig.csv').set_index('label')['oll_pos_mean']
print(f'{"label":18s} GradCAM   IG     (diff)')
for l in g.index:
    if l in i.index:
        print(f'  {l:18s} {g[l]:.3f}   {i[l]:.3f}  ({i[l]-g[l]:+.3f})')
# rank agreement is the headline: do both methods order the diseases the same?
print('\nGrad-CAM ranking:', list(g.sort_values(ascending=False).index))
print('IG ranking:      ', list(i.sort_values(ascending=False).index))

### 7. Zip the two OLL CSVs + download → send to Jonathan

In [ ]:
!cd results && zip -q /content/oll_crosscheck.zip oll_by_disease.csv oll_by_disease_ig.csv && echo done
from google.colab import files
files.download('/content/oll_crosscheck.zip')